In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sp             
import scipy.sparse as sparse
import anndata as ad

In [2]:
adata = ad.read_h5ad("../data_interim/TMS_checked.h5ad")
print(adata.obs.columns)
unique_tissues = adata.obs['tissue'].unique()
for tissue in unique_tissues:
    # 【关键修复】清理 tissue 名称，将 / \ 空格等非法文件名字符替换为下划线
    # 这样 "spleen/marrow" 就会变成 "spleen_marrow"
    safe_tissue_name = re.sub(r'[\\/*?:"<>|\s]', "_", str(tissue))
    
    adata_tissue = adata[adata.obs['tissue'] == tissue].copy()
    
    # 使用 pathlib 拼接路径，自动处理跨平台的斜杠问题
    out_path = f'../data_interim/adata_{safe_tissue_name}.h5ad'
    
    adata_tissue.write(out_path)
    print(f"已保存: {out_path} (n_obs={adata_tissue.n_obs}, n_vars={adata_tissue.n_vars})")


Index(['age', 'cell', 'cell_ontology_class', 'cell_ontology_id',
       'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue',
       'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden'],
      dtype='object')
已保存: ../data_interim/adata_spleen_marrow.h5ad (n_obs=75938, n_vars=20138)
已保存: ../data_interim/adata_Fat.h5ad (n_obs=6777, n_vars=20138)
已保存: ../data_interim/adata_Liver.h5ad (n_obs=7294, n_vars=20138)
已保存: ../data_interim/adata_Lung.h5ad (n_obs=24540, n_vars=20138)
已保存: ../data_interim/adata_Kidney.h5ad (n_obs=21647, n_vars=20138)
已保存: ../data_interim/adata_Heart_and_Aorta.h5ad (n_obs=8613, n_vars=20138)


In [3]:
# ================= 配置区 =================
input_dir = Path('../data_interim')
output_dir = Path('../data_interim')
output_dir.mkdir(parents=True, exist_ok=True)

# 字段转换日志
conversion_log = []


def add_log(dataset, operation, old_field, new_field, status):
    conversion_log.append({
        "dataset": dataset,
        "operation": operation,
        "old_field": old_field,
        "new_field": new_field,
        "status": status
    })


def standardize_adata(adata, dataset_name):
    print(f"原始: {adata.n_obs} cells × {adata.n_vars} genes")
    print(list(adata.obs.columns))

    # 1. mouse_id_std
    if 'mouse.id' in adata.obs.columns:
        adata.obs['mouse_id_std'] = (
            adata.obs['mouse.id']
            .astype(str)
            .str.replace(r'[^a-zA-Z0-9\-]', '_', regex=True)
            .str.replace(r'_+', '_', regex=True)
            .str.strip('_')
            .astype('category')
        )
        add_log(dataset_name, "standardize", "mouse.id", "mouse_id_std", "success")

    # 2. tissue_std
    if 'tissue' in adata.obs.columns:
        adata.obs['tissue_std'] = (
            adata.obs['tissue']
            .astype(str)
            .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)
            .str.replace(r'_+', '_', regex=True)
            .str.strip('_')
            .str.lower()
            .astype('category')
        )
        add_log(dataset_name, "standardize", "tissue", "tissue_std", "success")

    # 3. cell_type_original
    for col in ['cell_ontology_class', 'cell_type', 'celltype']:
        if col in adata.obs.columns:
            adata.obs['cell_type_original'] = adata.obs[col].astype('category')
            add_log(dataset_name, "standardize", col, "cell_type_original", "success")
            break

    # 4. age_months
    if 'age' in adata.obs.columns:
        adata.obs['age_months'] = (
            adata.obs['age']
            .astype(str)
            .str.extract(r'(\d+)')
            .astype(float)
        )
        add_log(dataset_name, "convert", "age", "age_months", "success")

    # 5. age_group
    if 'age_months' in adata.obs.columns:
        def age_group(x):
            if pd.isna(x):
                return "unknown"
            elif x <= 6:
                return "young"
            elif x <= 18:
                return "middle"
            else:
                return "old"

        adata.obs['age_group'] = (
            adata.obs['age_months']
            .apply(age_group)
            .astype('category')
        )

    # 6. gene重复检查
    duplicate_genes = adata.var_names[adata.var_names.duplicated()]
    if len(duplicate_genes):
        add_log(dataset_name, "gene_check", "var_names", "duplicate_genes", f"{len(duplicate_genes)} duplicates")
        adata.var_names_make_unique()
    else:
        add_log(dataset_name, "gene_check", "var_names", "duplicate_genes", "none")

    # 7. Ensembl / gene symbol 检查
    sample = list(adata.var_names[:1000])
    ens_pattern = re.compile(r'^ENS(MUS)?G\d+')
    ens_ratio = sum(bool(ens_pattern.match(str(x))) for x in sample) / len(sample)

    if ens_ratio > 0.8:
        gene_type = "Ensembl_ID"
    elif ens_ratio < 0.2:
        gene_type = "Gene_symbol"
    else:
        gene_type = "Mixed"

    add_log(dataset_name, "gene_annotation", "var_names", gene_type, "checked")

    # 8. counts完整性
    if "counts" in adata.layers:
        X = adata.layers["counts"]
        count_source = "layers[counts]"
    else:
        X = adata.X
        count_source = "adata.X"

    if sparse.issparse(X):       # ✅ 使用 sparse.issparse
        values = X.data
    else:
        values = X.flatten()

    integer_counts = np.all(values.astype(int) == values)
    nan_check = np.isnan(values).any()

    add_log(
        dataset_name,
        "counts_check",
        count_source,
        "raw_counts",
        f"integer={integer_counts};nan={nan_check}"
    )

    return adata


# ================= 主流程 =================
h5ad_files = sorted(input_dir.glob("adata_*.h5ad"))

for file_path in h5ad_files:
    print("=" * 60)
    print(file_path.name)

    adata = sp.read_h5ad(file_path)    # ✅ sp = scanpy

    adata = standardize_adata(adata, file_path.stem)

    out_path = output_dir / (file_path.stem + "_standardized.h5ad")
    adata.write(out_path)
    print("保存:", out_path)

print("全部完成")

adata_Fat.h5ad
原始: 6777 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\adata_Fat_standardized.h5ad
adata_Heart_and_Aorta.h5ad
原始: 8613 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\adata_Heart_and_Aorta_standardized.h5ad
adata_Kidney.h5ad
原始: 21647 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\adata_Kidney_standardized.h5ad
adata_Liver.h5ad
原始: 7294 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_on

In [4]:
docs_dir = Path("../docs")
docs_dir.mkdir(parents=True, exist_ok=True)

# 1. 确保每个细胞具有唯一 cell_id
if "cell_id" not in adata.obs.columns:
    adata.obs["cell_id"] = adata.obs_names.astype(str)

assert adata.obs["cell_id"].is_unique, "cell_id存在重复"

# 2. metadata_mapping.tsv
metadata_records = []

for col in adata.obs.columns:
    metadata_records.append({
        "field": col,
        "dtype": str(adata.obs[col].dtype),
        "n_unique": int(adata.obs[col].nunique()),
        "missing_count": int(adata.obs[col].isna().sum())
    })

metadata_mapping = pd.DataFrame(metadata_records)
metadata_mapping.to_csv(
    docs_dir / "metadata_mapping.tsv",
    sep="\t",
    index=False
)

print("已生成 metadata_mapping.tsv")


# 3. gene_id_check.tsv
gene_records = []

for idx, gene in enumerate(adata.var_names):
    gene_records.append({
        "gene_index": idx,
        "gene_id": gene,
        "is_ensembl": bool(str(gene).startswith("ENSMUSG"))
    })

gene_check = pd.DataFrame(gene_records)

gene_check.to_csv(
    docs_dir / "gene_id_check.tsv",
    sep="\t",
    index=False
)

print("已生成 gene_id_check.tsv")


# 4. counts来源与完整性检查
count_source = "unknown"

if "counts" in adata.layers:
    count_source = "layers[counts]"
    counts = adata.layers["counts"]
else:
    count_source = "adata.X"
    counts = adata.X

if hasattr(counts, "data"):
    values = counts.data
else:
    values = np.asarray(counts).ravel()

counts_report = pd.DataFrame([{
    "count_source": count_source,
    "min_value": float(values.min()),
    "max_value": float(values.max()),
    "negative_values": int(np.sum(values < 0)),
    "nan_values": int(np.sum(pd.isna(values))),
    "integer_like": bool(np.all(np.isclose(values, np.round(values))))
}])

counts_report.to_csv(
    docs_dir / "counts_integrity_check.tsv",
    sep="\t",
    index=False
)

print("已生成 counts_integrity_check.tsv")


# 5. 标准化对象汇总报告
summary = pd.DataFrame([{
    "n_cells": adata.n_obs,
    "n_genes": adata.n_vars,
    "n_mouse": adata.obs["mouse_id_std"].nunique()
        if "mouse_id_std" in adata.obs.columns else np.nan,
    "n_tissue": adata.obs["tissue_std"].nunique()
        if "tissue_std" in adata.obs.columns else np.nan,
    "has_cell_id": "cell_id" in adata.obs.columns,
    "has_mouse_id": "mouse_id_std" in adata.obs.columns
}])

summary.to_csv(
    docs_dir / "standardization_report.tsv",
    sep="\t",
    index=False
)

print("第3周标准化审计补充完成")


已生成 metadata_mapping.tsv
已生成 gene_id_check.tsv
已生成 counts_integrity_check.tsv
第3周标准化审计补充完成
